# Ingest plan — programmatic fetch (StatCan WDS + CMHC)

This notebook searches StatCan's Web Data Service (WDS) for relevant product IDs (PIDs) and scrapes CMHC table pages for CSV links, then downloads matching CSVs to `data/raw/`. Run cells in order.

In [34]:
# Cell 1: setup and imports
import requests
import re
import json
import time
import zipfile
import shutil
from pathlib import Path
import pandas as pd

PROJECT = Path('.')
RAW = PROJECT / 'data' / 'raw'
PROCESSED = PROJECT / 'data' / 'processed'
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# StatCan WDS base endpoints (see https://www150.statcan.gc.ca/t1/wds)
STATCAN_WDS_BASE = 'https://www150.statcan.gc.ca/t1/wds'
ALL_CUBES_LITE = f'{STATCAN_WDS_BASE}/rest/getAllCubesListLite'
FULL_TABLE_CSV = f'{STATCAN_WDS_BASE}/rest/getFullTableDownloadCSV'  # append /{PID}/en

# Simple retrying HTTP helper to handle transient 404/403/timeouts
SESSION = requests.Session()
DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept": "*/*",
}

def request_with_retry(url, *, method="GET", retries=3, backoff=2, timeout=30, **kwargs):
    headers = kwargs.pop("headers", {})
    merged_headers = {**DEFAULT_HEADERS, **headers}
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            resp = SESSION.request(method=method, url=url, headers=merged_headers, timeout=timeout, **kwargs)
            if resp.status_code in (403, 404) and attempt < retries:
                time.sleep(backoff * attempt)
                continue
            resp.raise_for_status()
            return resp
        except Exception as exc:
            last_exc = exc
            if attempt == retries:
                break
            time.sleep(backoff * attempt)
    raise last_exc

print('setup complete')

setup complete


# Note on WDS API Status

The StatCan WDS (Web Data Service) API has a confusing method for gaining access to the zip files. 

1. **Primary method**: Direct table download using known Product IDs (PIDs) via the `getFullTableDownloadCSV` endpoint
2. **Fallback method**: WDS search API (if available) to discover tables by keyword

Each StatCan dataset in Cell 4 now includes a `pid` field with the table's Product ID. This allows the notebook to download data directly without relying on the search API. If a download fails, the notebook will provide a manual download link.

For manual downloads, visit: https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=YOUR-PID-HERE

In [35]:
# Cell 2: helpers for StatCan WDS
_statcan_cubes_cache = None

def fetch_all_statcan_cubes(force=False):
    global _statcan_cubes_cache
    if _statcan_cubes_cache is not None and not force:
        return _statcan_cubes_cache
    try:
        resp = request_with_retry(ALL_CUBES_LITE, method="POST", retries=4, backoff=2, timeout=45, json={})
    except Exception as first_exc:
        # Fallback to GET if POST is rejected (some StatCan edges return 404 on POST)
        resp = request_with_retry(ALL_CUBES_LITE, method="GET", retries=4, backoff=2, timeout=45)
    data = resp.json()
    # data expected as list of cube metadata objects with 'title' and 'pid'
    _statcan_cubes_cache = data
    return data

def statcan_search(keyword, max_results=10):
    keyword = keyword.lower()
    cubes = fetch_all_statcan_cubes()
    matches = []
    for c in cubes:
        title = (c.get('title') or c.get('productTitle') or c.get('cubeTitleEn') or '')
        pid = c.get('pid') or c.get('productId') or c.get('productID')
        if not title or not pid:
            continue
        if keyword in title.lower():
            matches.append({'pid': str(pid), 'title': title})
        if len(matches) >= max_results:
            break
    return matches

def statcan_zip_candidates(wds_object_url, lang='en'):
    """Return download candidates, inserting /{lang}/ when WDS omits it."""
    if not wds_object_url:
        return []
    urls = [wds_object_url]
    bare_segment = "/n1/tbl/csv/"
    if re.search(r"https://www150\\.statcan\\.gc\\.ca/n1/tbl/csv/", wds_object_url):
        normalized = wds_object_url.replace(bare_segment, f"/n1/{lang}/tbl/csv/")
        urls.append(normalized)
    # Preserve order, drop dupes
    return list(dict.fromkeys(urls))

def download_statcan_zip_with_fallback(urls, dest_path):
    """Attempt each candidate until one succeeds."""
    dest = Path(dest_path)
    headers = {**DEFAULT_HEADERS, "Accept": "*/*"}
    last_error = None
    for candidate in urls:
        try:
            print('downloading', candidate)
            with SESSION.get(candidate, headers=headers, stream=True, timeout=90, allow_redirects=True) as resp:
                if resp.status_code == 404:
                    raise requests.HTTPError("404 Not Found", response=resp)
                resp.raise_for_status()
                with open(dest, 'wb') as fh:
                    for chunk in resp.iter_content(1024 * 1024):
                        if chunk:
                            fh.write(chunk)
            print('wrote', dest)
            return dest
        except Exception as exc:
            last_error = exc
            try:
                dest.unlink()
            except FileNotFoundError:
                pass
            continue
    raise RuntimeError(f'All StatCan download candidates failed. Last error: {last_error}')

def download_statcan_full_table_zip(pid, dest_path, lang='en'):
    dest = Path(dest_path)
    if dest.exists():
        print(dest, 'exists')
        return dest

    normalized_pid = normalize_statcan_pid(pid)
    if not normalized_pid:
        raise ValueError(f'Unable to normalize StatCan PID: {pid}')

    manifest_url = f'{FULL_TABLE_CSV}/{normalized_pid}/{lang}'
    print('requesting manifest', manifest_url)
    try:
        resp = request_with_retry(manifest_url, method="POST", retries=4, backoff=2, timeout=90, json={})
    except Exception:
        resp = request_with_retry(manifest_url, method="GET", retries=4, backoff=2, timeout=90)

    data = resp.json()
    object_url = None
    if isinstance(data, dict):
        status = (data.get('status') or '').upper()
        if status and status != 'SUCCESS':
            raise ValueError(f"StatCan manifest error for {pid}: {data}")
        object_url = data.get('object')
    elif isinstance(data, list) and data:
        object_url = data[0].get('object')
    if not object_url:
        raise ValueError(f'No download object URL returned for {pid}: {data}')

    zip_candidates = statcan_zip_candidates(object_url, lang=lang)
    if not zip_candidates:
        raise ValueError(f'Unable to build StatCan download candidates for {pid}')

    zip_path = dest.with_suffix(dest.suffix + '.zip')
    download_statcan_zip_with_fallback(zip_candidates, zip_path)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        csv_members = [name for name in zip_ref.namelist() if name.lower().endswith('.csv')]
        if not csv_members:
            raise ValueError(f'Zip archive for {pid} does not contain a CSV file: {zip_ref.namelist()}')
        csv_member = csv_members[0]
        with zip_ref.open(csv_member) as src, open(dest, 'wb') as dst:
            shutil.copyfileobj(src, dst)
    try:
        zip_path.unlink()
    except FileNotFoundError:
        pass
    print('wrote', dest)
    return dest

def normalize_statcan_pid(pid_value):
    """Accepts table numbers like 98-10-0009-01 and returns the 8-digit productId."""
    if pid_value is None:
        return None
    digits = ''.join(ch for ch in str(pid_value) if ch.isdigit())
    if len(digits) == 8:
        return digits
    if len(digits) > 8:
        normalized = digits[-8:]
        if str(pid_value) != normalized:
            print(f"  Normalized StatCan PID {pid_value} -> {normalized}")
        return normalized
    return None

import re, requests
from bs4 import BeautifulSoup
from urllib.parse import quote

def harvest_pids_from_site(query: str, max_results: int = 10) -> list[dict]:
    url = f"https://www150.statcan.gc.ca/n1/en/type/data?text={quote(query)}"
    html = requests.get(url, timeout=30, headers={"User-Agent":"Mozilla/5.0"}).text
    soup = BeautifulSoup(html, "html.parser")

    out = []
    for a in soup.select("a[href]"):
        href = a["href"]
        m = re.search(r"pid=(\d{8})", href)
        if m:
            title = a.get_text(" ", strip=True)[:200]
            out.append({"pid": m.group(1), "title": title, "href": href})
    # de-dupe preserving order
    seen, deduped = set(), []
    for row in out:
        if row["pid"] not in seen:
            seen.add(row["pid"])
            deduped.append(row)
    return deduped[:max_results]


In [36]:
# Cell 3: helpers for CMHC page scraping and generic downloading
import re
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup

UA_HEADERS = {"User-Agent": "Mozilla/5.0"}

def find_cmhc_xlsx_links_on_page(page_url: str) -> list[str]:
    html = requests.get(page_url, headers=UA_HEADERS, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")

    links = []

    def consider(raw_url: str | None):
        if not raw_url:
            return
        full = urljoin(page_url, raw_url.strip())
        parsed_path = urlparse(full).path.lower()
        if parsed_path.endswith('.xlsx'):
            links.append(full)

    for a in soup.select("a[href]"):
        consider(a.get("href"))

    # Some CMHC pages (e.g., housing starts) stash the XLSX path inside hidden inputs
    for input_tag in soup.select('input[value]'):
        consider(input_tag.get('value'))

    # de-dupe preserving order
    out = []
    for l in links:
        if l not in out:
            out.append(l)
    return out

def choose_latest_cmhc_link(links: list[str]) -> str | None:
    """
    Prefer links that contain /YYYY/ in the path, pick max year.
    If no year is present, fall back to the first link.
    """
    if not links:
        return None

    def year_key(url: str) -> int:
        m = re.search(r"/(20\d{2})/", url)
        return int(m.group(1)) if m else -1

    return sorted(links, key=year_key, reverse=True)[0]

def download_file_guarded(url: str, dest_path):
    with requests.get(url, headers=UA_HEADERS, stream=True, timeout=60, allow_redirects=True) as r:
        r.raise_for_status()
        ctype = (r.headers.get("Content-Type") or "").lower()

        # If we're expecting an Excel file, block HTML downloads.
        if "text/html" in ctype:
            snippet = r.text[:200]
            raise RuntimeError(f"Downloaded HTML instead of file. url={url} content-type={ctype} snippet={snippet!r}")

        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)


def find_data_links_on_page(page_url):
    try:
        r = request_with_retry(page_url, retries=4, backoff=2, timeout=45)
    except Exception as e:
        print('failed to fetch', page_url, e)
        return []
    html = r.text
    pattern = r"href=[\"']([^\"']+)[\"']"
    hrefs = re.findall(pattern, html, flags=re.I)
    valid_suffixes = ('.csv', '.xlsx', '.xls')
    candidates = []
    for h in hrefs:
        lowered = h.lower()
        has_suffix = lowered.endswith(valid_suffixes)
        looks_like_download = 'download' in lowered and any(ext in lowered for ext in valid_suffixes)
        if has_suffix or looks_like_download:
            if h.startswith('//'):
                h = 'https:' + h
            elif h.startswith('/'):
                base = re.match(r'(https?://[^/]+)', page_url)
                if base:
                    h = base.group(1) + h
            candidates.append(h)
    seen = []
    out = []
    for c in candidates:
        if c not in seen:
            seen.append(c)
            out.append(c)
    return out

def download_file(url, dest_path):
    dest = Path(dest_path)
    if dest.exists():
        print(dest, 'exists')
        return dest
    print('downloading', url)
    r = request_with_retry(url, retries=4, backoff=2, timeout=90, stream=True)
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(1024*1024):
            if chunk:
                f.write(chunk)
    print('wrote', dest)
    return dest


In [37]:
# Cell 4: dataset definitions (keywords / CMHC pages / direct PIDs)
# StatCan PIDs are scraped at runtime so downstream cells always see fresh identifiers
datasets = [
    {
        'name': 'median_household_income',
        'provider': 'statcan',
        'keyword': 'median total income',
        'pid': None,
        'description': 'Median total income of families and persons',
    },
    {
        'name': 'population_estimates',
        'provider': 'statcan',
        'keyword': 'population estimates cma',
        'pid': None,
        'description': 'Population estimates, quarterly',
    },
    {
        'name': 'unemployment_rate',
        'provider': 'statcan',
        'keyword': 'unemployment rate cma',
        'pid': None,
        'description': 'Labour force characteristics by census metropolitan area',
    },
    {
        'name': 'cpi_all_items',
        'provider': 'statcan',
        'keyword': 'consumer price index all-items',
        'pid': None,
        'description': 'Consumer Price Index, monthly',
    },
    {
        # CMHC rental market tables live on a landing page that publishes annual XLSX assets
        'name': 'rental_market_rents',
        'provider': 'cmhc',
        'page': 'https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/rental-market/rental-market-report-data-tables',
        'direct_url': None,
        'fallback_direct_url': 'https://assets.cmhc-schl.gc.ca/sites/cmhc/professional/housing-markets-data-and-research/housing-data-tables/rental-market/rental-market-report-data-tables/2025/rmr-canada-2025-en.xlsx?rev=11bbba5e-64a5-4dcd-a81f-7252c2b12537&_gl=1*529jeb*_gcl_au*NzkzNDcyMTcxLjE3NjY2MTc2NDU.*_ga*NTc3NDE3MTQyLjE3NjY2MTc2NDY.*_ga_CY7T7RT5C4*czE3NjY2MTc2NDYkbzEkZzEkdDE3NjY2MTc2NzQkajMyJGwwJGgw',
        'description': 'CMHC Rental Market Report data tables',
    },
    {
        'name': 'housing_starts',
        'provider': 'cmhc',
        'page': 'https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/data-tables/housing-market-data/monthly-housing-starts-construction-data-tables',
        'direct_url': None,
        'fallback_direct_url': None,
        'description': 'Monthly housing starts and construction data tables',
    },
]


def populate_statcan_pids(items, *, max_results=5):
    """Use site scraping (with WDS keyword fallback) to set StatCan PIDs in-place."""
    for ds in items:
        if ds.get('provider') != 'statcan':
            continue
        keyword = (ds.get('keyword') or ds.get('name') or '').strip()
        if not keyword:
            raise ValueError(f"Dataset {ds.get('name')} missing keyword for StatCan PID discovery")
        scraped = []
        scrape_error = None
        try:
            scraped = harvest_pids_from_site(keyword, max_results=max_results)
        except Exception as exc:
            scrape_error = exc
        if scraped:
            candidate = scraped[0]
            ds['pid'] = normalize_statcan_pid(candidate['pid'])
            ds['pid_source'] = 'harvest'
            ds['pid_title'] = candidate.get('title')
            continue
        search_hits = statcan_search(keyword, max_results=1)
        if search_hits:
            ds['pid'] = normalize_statcan_pid(search_hits[0]['pid'])
            ds['pid_source'] = 'wds_keyword'
            ds['pid_title'] = search_hits[0].get('title')
            continue
        error_msg = f"Unable to auto-discover StatCan PID for dataset '{ds.get('name')}' with keyword '{keyword}'."
        if scrape_error:
            error_msg += f" Scrape error: {scrape_error}"
        raise ValueError(error_msg)
    return items


def populate_cmhc_direct_urls(items):
    """Scrape CMHC landing pages to attach direct download URLs in-place, with fallbacks."""
    for ds in items:
        if ds.get('provider') != 'cmhc':
            continue
        if ds.get('direct_url'):
            continue
        page = ds.get('page')
        fallback_url = ds.get('fallback_direct_url')
        if not page and not fallback_url:
            raise ValueError(f"CMHC dataset '{ds.get('name')}' missing landing page URL or fallback for scraping")
        scrape_notes = []
        xlsx_links = []
        xlsx_error = None
        if page:
            try:
                xlsx_links = find_cmhc_xlsx_links_on_page(page)
            except Exception as exc:
                xlsx_error = exc
                scrape_notes.append(f"XLSX scrape error: {exc}")
            if xlsx_links:
                selected = choose_latest_cmhc_link(xlsx_links)
                if selected:
                    ds['direct_url'] = selected
                    ds['cmhc_link_source'] = 'xlsx_latest'
                    ds['cmhc_notes'] = '; '.join(scrape_notes) if scrape_notes else 'Selected latest XLSX link from CMHC scrape'
                    continue
            generic_links = []
            try:
                generic_links = find_data_links_on_page(page)
            except Exception as exc:
                scrape_notes.append(f"Generic scrape error: {exc}")
            candidates = [link for link in generic_links if link.lower().endswith(('.csv', '.xlsx', '.xls', '.zip'))]
            if candidates:
                ds['direct_url'] = candidates[0]
                ds['cmhc_link_source'] = 'generic'
                note = 'Selected first generic data link from CMHC page'
                if scrape_notes:
                    note = f"{note}; {'; '.join(scrape_notes)}"
                ds['cmhc_notes'] = note
                continue
        if fallback_url:
            ds['direct_url'] = fallback_url
            ds['cmhc_link_source'] = 'fallback'
            fallback_note = 'Used configured CMHC fallback URL'
            if scrape_notes:
                fallback_note = f"{fallback_note}; {'; '.join(scrape_notes)}"
            ds['cmhc_notes'] = fallback_note
            continue
        # Mark manual follow-up rather than raising so the download summary can report it
        manual_note = 'No CMHC download link discovered automatically; manual follow-up required'
        if scrape_notes:
            manual_note = f"{manual_note} ({'; '.join(scrape_notes)})"
        ds['cmhc_manual_required'] = True
        ds['cmhc_link_source'] = 'manual'
        ds['cmhc_notes'] = manual_note
    return items


datasets = populate_statcan_pids(datasets)
datasets = populate_cmhc_direct_urls(datasets)
print('datasets defined with StatCan PIDs and CMHC download URLs populated')

failed to fetch https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/rental-market/rental-market-report-data-tables 404 Client Error: Page not found for url: https://www.cmhc-schl.gc.ca/404
datasets defined with StatCan PIDs and CMHC download URLs populated


## Downloading the defined datasets

Run this section to loop through each dataset, attempt a programmatic download, and capture a status report for any manual follow-up that might be needed.

In [38]:
# Cell 5: execute downloads and summarize outcomes
from datetime import datetime

# simple helper so filenames are filesystem-safe
_filename_pattern = re.compile(r"[^A-Za-z0-9_.-]+")

def sanitize_filename(value):
    return _filename_pattern.sub('_', value).strip('_') or 'dataset'


def pick_extension(url, default='.csv'):
    base = url.split('?', 1)[0]
    suffix = Path(base).suffix
    return suffix if suffix else default


summary_rows = []
start_ts = datetime.utcnow().isoformat()

for ds in datasets:
    record = {
        'name': ds.get('name'),
        'provider': ds.get('provider'),
        'status': 'pending',
        'path': None,
        'manual_url': None,
        'details': '',
    }
    statcan_manual_url = None
    statcan_lang = ds.get('lang', 'en')
    try:
        if ds.get('provider') == 'statcan':
            pid = normalize_statcan_pid(ds.get('pid'))
            note_parts = []
            if not pid and ds.get('keyword'):
                try:
                    scraped = harvest_pids_from_site(ds['keyword'], max_results=3)
                except Exception as scrape_exc:
                    scraped = []
                    note_parts.append(f"Scrape error: {scrape_exc}")
                if scraped:
                    primary = scraped[0]
                    pid = normalize_statcan_pid(primary['pid'])
                    title = primary.get('title')
                    title_suffix = f" — {title}" if title else ''
                    note_parts.append(f"Scraped PID {pid}{title_suffix}")
            if not pid and ds.get('keyword'):
                results = statcan_search(ds['keyword'], max_results=1)
                if results:
                    pid = normalize_statcan_pid(results[0]['pid'])
                    note_parts.append(f"Keyword search matched PID {pid}")
            if not pid:
                raise ValueError('StatCan dataset missing PID after scrape + keyword search attempts')
            lang_suffix = 'eng' if statcan_lang == 'en' else 'fra'
            statcan_manual_url = f"https://www150.statcan.gc.ca/n1/{statcan_lang}/tbl/csv/{pid}-{lang_suffix}.zip"
            dest = RAW / f"{sanitize_filename(ds['name'])}_{pid}.csv"
            download_statcan_full_table_zip(pid, dest, lang=statcan_lang)
            record['status'] = 'downloaded'
            record['path'] = str(dest)
            manual_url = f"https://www150.statcan.gc.ca/t1/tbl1/{statcan_lang}/tv.action?pid={pid}"
            record['manual_url'] = manual_url
            record['details'] = '; '.join(note_parts) if note_parts else f"Fetched via PID {pid}"
        elif ds.get('provider') == 'cmhc':
            target_url = ds.get('direct_url')
            scrape_notes = ds.get('cmhc_notes') or ''
            if not target_url:
                record['status'] = 'manual'
                record['manual_url'] = ds.get('page') or ds.get('fallback_direct_url')
                record['details'] = scrape_notes or 'Manual CMHC download required; no automated link found'
                summary_rows.append(record)
                continue
            dest = RAW / f"{sanitize_filename(ds['name'])}{pick_extension(target_url)}"
            download_file_guarded(target_url, dest)
            record['status'] = 'downloaded'
            record['path'] = str(dest)
            record['manual_url'] = ds.get('page') or target_url
            link_source = ds.get('cmhc_link_source')
            note_chunks = []
            if link_source:
                note_chunks.append(f"CMHC link source: {link_source}")
            if scrape_notes:
                note_chunks.append(scrape_notes)
            record['details'] = '; '.join(note_chunks) if note_chunks else 'Downloaded via direct CMHC URL'
        else:
            raise ValueError(f"Unknown provider '{ds.get('provider')}'")
    except Exception as exc:
        manual = ds.get('page') or ds.get('direct_url')
        if record['provider'] == 'statcan' and statcan_manual_url:
            manual = statcan_manual_url
        if record['manual_url'] and manual and record['manual_url'] != manual:
            record['manual_url'] = f"{record['manual_url']} | {manual}"
        else:
            record['manual_url'] = record['manual_url'] or manual
        record['status'] = 'error'
        record['details'] = (record['details'] + ('; ' if record['details'] else '') + str(exc)).strip()
    summary_rows.append(record)

summary_df = pd.DataFrame(summary_rows)
summary_df.insert(0, 'run_started_utc', start_ts)
summary_df

data/raw/median_household_income_10100139.csv exists
data/raw/population_estimates_10100139.csv exists
data/raw/unemployment_rate_10100139.csv exists
data/raw/cpi_all_items_10100139.csv exists


/tmp/ipykernel_5869/1742338806.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = datetime.utcnow().isoformat()


,run_started_utc,name,provider,status,path,manual_url,details
0,2026-01-03T21:04:28.204809,median_household_income,statcan,downloaded,data/raw/median_household_income_10100139.csv,https://www150.statcan.gc.ca/t1/tbl1/en/tv.act...,Fetched via PID 10100139
1,2026-01-03T21:04:28.204809,population_estimates,statcan,downloaded,data/raw/population_estimates_10100139.csv,https://www150.statcan.gc.ca/t1/tbl1/en/tv.act...,Fetched via PID 10100139
2,2026-01-03T21:04:28.204809,unemployment_rate,statcan,downloaded,data/raw/unemployment_rate_10100139.csv,https://www150.statcan.gc.ca/t1/tbl1/en/tv.act...,Fetched via PID 10100139
3,2026-01-03T21:04:28.204809,cpi_all_items,statcan,downloaded,data/raw/cpi_all_items_10100139.csv,https://www150.statcan.gc.ca/t1/tbl1/en/tv.act...,Fetched via PID 10100139
4,2026-01-03T21:04:28.204809,rental_market_rents,cmhc,downloaded,data/raw/rental_market_rents.xlsx,https://www.cmhc-schl.gc.ca/professionals/hous...,CMHC link source: fallback; Used configured CM...
5,2026-01-03T21:04:28.204809,housing_starts,cmhc,downloaded,data/raw/housing_starts.xlsx,https://www.cmhc-schl.gc.ca/professionals/hous...,CMHC link source: xlsx_latest; Selected latest...
